**Status: exports the pipeline's tables to the static site's JSON files.** Run last in the weekly job (and in the Saturday job after `phase_1b_results`). Writes `/Volumes/workspace/default/f1_data/site_data/`, which the GitHub Action downloads and publishes. Results only go into each race's `result.json`, never into files the site loads up front.

In [ ]:
import datetime
import json
import os
import shutil
import sys
import tempfile
import time
import urllib.request

sys.path.insert(0, os.path.abspath(".."))
from site_export import export as ex

OUT = "/Volumes/workspace/default/f1_data/site_data"
TABLES = ["f1_replay_lap_state", "f1_race_results", "f1_live_predictions", "f1_battle_pairs", "f1_podium_prior_oof", "f1_podium_model",
          "f1_podium_driver_form", "f1_podium_team_form", "f1_podium_circuits", "f1_podium_recent_grids", "f1_live_backtest",
          "f1_podium_backtest", "f1_battle_bands"]

In [ ]:
# season calendar with session times (Jolpica)
def schedule(year):
    for attempt in range(3):
        try:
            with urllib.request.urlopen(f"https://api.jolpi.ca/ergast/f1/{year}.json?limit=100", timeout=60) as r:
                return ex.calendar_from_jolpica(json.load(r))
        except Exception as e:
            print(f"retry {year}: {e}")
            time.sleep(10 * (attempt + 1))
    return []

this_year = datetime.date.today().year
calendar = schedule(this_year) or schedule(this_year - 1)
print(len(calendar), "calendar rounds")

In [ ]:
tables = {t: spark.table(f"workspace.default.{t}").toPandas() for t in TABLES}
tmp = tempfile.mkdtemp()
summary = ex.write_site_data(tables, calendar, tmp, datetime.date.today().isoformat())
if os.path.isdir("/Volumes/workspace/default/f1_data/live_models"):
    shutil.copytree("/Volumes/workspace/default/f1_data/live_models", os.path.join(tmp, "livemodels"))
shutil.rmtree(OUT, ignore_errors=True)
shutil.copytree(tmp, OUT)
print(f"Exported {summary['races']} races, {summary['files']} files")